In [ ]:
!date

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/'

In [ ]:
mct_adata = sc.read_h5ad(f'{projdir}/h5ad/chrom100k/raw_norm.h5ad')

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/snm3C/mc/'
m3c_adata = sc.read_h5ad(f'{indir}/h5ad/chrom100k/raw_norm.h5ad')

In [ ]:
mct_adata

In [ ]:
m3c_adata

### show that there is a methylation difference

In [ ]:
m3c_meta = pd.read_csv(f'{projdir}/csv/choo_raw/B01_m3C_filtered.csv')
mct_meta = pd.read_csv(f'{projdir}/csv/choo_raw/B01_mCT_filtered.csv')
m3c_meta.shape, mct_meta.shape

In [ ]:
mct_meta.index = mct_meta['wellprefix']
m3c_meta.index = m3c_meta['wellprefix']

In [ ]:
keep = ['time', 'line', 'DNA_mCGfrac', 'DNA_mCHfrac', 'DNA_mCCCfrac']
tmp_mct = mct_meta[keep]

In [ ]:
keep = ['time', 'line','mCGfrac', 'mCHfrac', 'mCCCfrac']
tmp_m3c = m3c_meta[keep]

In [ ]:
tmp_mct.columns = ['time', 'line','mCGfrac', 'mCHfrac', 'mCCCfrac']

In [ ]:
tmp_mct['assay'] = 'snmCT'
tmp_m3c['assay'] = 'snm3C'

In [ ]:
meta = pd.concat([tmp_mct, tmp_m3c])

In [ ]:
meta.head()

In [ ]:
meta['donor_time'] = [f'{x}_{y}' for x,y in zip(meta['line'], meta['time'])]

In [ ]:
tmp_meta = meta.sort_values(by=['time', 'line'])

In [ ]:
### probably better to make the same ridgeplot that choo made?
to_plot = pd.melt(tmp_meta, id_vars=['donor_time', 'assay'], value_vars='mCGfrac')
to_plot.head()

In [ ]:
fig, axes = plt.subplots(1, figsize=(12, 8))

x_lab = 'donor_time'
y_lab = 'value'
hue = 'assay'

ax = sns.boxplot(data=to_plot, x=x_lab, y=y_lab, hue=hue, showfliers=False)
ax.set_xlabel('Donor', fontsize=16)
ax.set_ylabel('# droplets', fontsize=16)
ax.tick_params(axis='x', labelsize=14, labelrotation=45)

sns.move_legend(ax, "upper left", bbox_to_anchor=(1.05, 1))

In [ ]:
to_plot['donor'] = [x.split('_')[0] for x in to_plot['donor_time']]
to_plot['time'] = [int(x.split('_')[1]) for x in to_plot['donor_time']]

In [ ]:
to_plot.to_csv(f'{projdir}/csv/integrate_m3c/ridgeplot_mc.csv', sep='\t', header=True, index=True)

### subset to the subsampled!

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/'
merged = sc.read_h5ad(f'{indir}/rna/h5ad/processed/merged.h5ad')
snmct = merged[merged.obs['assay'] == 'snmct']

In [ ]:
snmct

In [ ]:
mct_adata.obs.index = [x.replace('_indexed', '') for x in mct_adata.obs.index]

In [ ]:
mct_adata = mct_adata[mct_adata.obs.index.isin(snmct.obs.index)]

### merge and select top 20,000 HVFs

In [ ]:
m3c_adata.obs.index = [x.replace('_indexed', '') for x in m3c_adata.obs.index]

In [ ]:
mct_adata.obs = tmp_mct.reindex(mct_adata.obs.index)
m3c_adata.obs = tmp_m3c.reindex(m3c_adata.obs.index)

In [ ]:
adata = mct_adata.concatenate(m3c_adata)

In [ ]:
adata

In [ ]:
adata = adata[~adata.obs['time'].isna()]

In [ ]:
adata.write_h5ad(f'{projdir}/h5ad/integrate_m3c/merged.raw.subsample.h5ad')

### now let's do dim reduction

In [ ]:
adata = sc.read_h5ad(f'{projdir}/h5ad/integrate_m3c/merged.raw.h5ad')

In [ ]:
hvfs = 20000
sc.pp.highly_variable_genes(adata, n_top_genes = hvfs)

In [ ]:
sc.tl.pca(adata, n_comps = 50, use_highly_variable=True)

In [ ]:
sc.pl.pca(adata, components = ['1,2', '3,4', '5,6', '7,8'], color=['time', 'line', 'assay'])

In [ ]:
sc.pp.neighbors(adata, n_pcs=20, random_state=0)
sc.tl.leiden(adata, resolution=0.3, random_state=0)
sc.tl.umap(adata, random_state=0)

In [ ]:
sc.pl.umap(adata, color=['time', 'line'])

In [ ]:
sc.pl.umap(adata, color=['assay'])

In [ ]:
adata.write_h5ad(f'{projdir}/h5ad/integrate_m3c/merged.processed.h5ad')

### what about harmony?

In [ ]:
sc.external.pp.harmony_integrate(adata, key='assay')

In [ ]:
sc.pl.pca(adata, components = ['1,2', '3,4', '5,6', '7,8'], color=['time', 'line', 'assay'])

In [ ]:
sc.pp.neighbors(adata, n_pcs=20, random_state=0, use_rep='X_pca_harmony')
sc.tl.leiden(adata, resolution=0.3, random_state=0)
sc.tl.umap(adata, random_state=0)

In [ ]:
sc.pl.umap(adata, color=['time', 'line'])

In [ ]:
sc.pl.umap(adata, color=['assay'])

In [ ]:
adata.write_h5ad(f'{projdir}/h5ad/integrate_m3c/merged.processed.harmony.h5ad')

In [ ]:
!date